In [8]:
import pandas as pd
import numpy as np
import stanza
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Inicializácia Stanza
stanza.download('sk')
nlp = stanza.Pipeline(lang='sk', processors='tokenize')

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

2025-06-09 09:24:06 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-06-09 09:24:06 INFO: Downloading default packages for language: sk (Slovak) ...
2025-06-09 09:24:07 INFO: File exists: C:\Users\marek\stanza_resources\sk\default.zip
2025-06-09 09:24:09 INFO: Finished downloading models and saved to C:\Users\marek\stanza_resources
2025-06-09 09:24:09 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-06-09 09:24:09 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-06-09 09:24:09 WARNING: Language sk package default expects mwt, which has been added
2025-06-09 09:24:09 INFO: Loading these models for language: sk (Slovak):
| Processor | Package |
-----------------------
| tokenize  | snk     |
| mwt       | snk     |

2025-06-09 09:24:09 INFO: Using device: cpu
2025-06-09 09:24:09 INFO: Loading: tokenize
2025-06-09 09:24:09 INFO: Loading: mwt
2025-06-09 09:24:09 INFO: Done loading processors!


In [10]:
df = pd.read_csv("data/contract_criteria_final_general_only.csv")

In [11]:
# Inicializácia Stanza
stanza.download('sk')
nlp = stanza.Pipeline(lang='sk', processors='tokenize')

# Načítanie modelu a dát
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Výber textových kritérií
all_criteria = df["criterion"].fillna("").astype(str).tolist()

# Výpočet embeddingov
all_embeddings = model.encode(all_criteria, convert_to_tensor=False)

# Uloženie
np.save("embeddings/criteria_embeddings_split.npy", all_embeddings)
df.to_csv("data/criteria_data.csv", index=False)

2025-06-09 09:35:49 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-06-09 09:35:49 INFO: Downloading default packages for language: sk (Slovak) ...
2025-06-09 09:35:50 INFO: File exists: C:\Users\marek\stanza_resources\sk\default.zip
2025-06-09 09:35:52 INFO: Finished downloading models and saved to C:\Users\marek\stanza_resources
2025-06-09 09:35:52 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-06-09 09:35:52 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-06-09 09:35:52 WARNING: Language sk package default expects mwt, which has been added
2025-06-09 09:35:52 INFO: Loading these models for language: sk (Slovak):
| Processor | Package |
-----------------------
| tokenize  | snk     |
| mwt       | snk     |

2025-06-09 09:35:52 INFO: Using device: cpu
2025-06-09 09:35:52 INFO: Loading: tokenize
2025-06-09 09:35:52 INFO: Loading: mwt
2025-06-09 09:35:52 INFO: Done loading processors!


In [12]:
# Výber textových popisov
all_descriptions = df["description"].fillna("").astype(str).tolist()

# Výpočet embeddingov
all_embeddings_d = model.encode(all_descriptions, convert_to_tensor=True)

# Uloženie
np.save("embeddings/descriptions_embeddings_split.npy", all_embeddings_d)
df.to_csv("data/descriptions_data.csv", index=False)

In [6]:
# import pandas as pd
# from transformers import pipeline

# # Načítanie dát
# df = pd.read_csv("data/contract_criteria_clean_split.csv")

# # Pipeline na generovanie textu
# generator = pipeline("text2text-generation", model="google/flan-t5-base", max_length=20)

# # Pomocná funkcia na vytvorenie kategórie
# def create_category(criterion, description):
#     prompt = (
#         f"Zadaj stručnú všeobecnú kategóriu pre nasledujúce kritérium verejného obstarávania:\n"
#         f"Kritérium: {criterion}\n"
#         f"Popis: {description}\n"
#         f"Kategória:"
#     )
#     response = generator(prompt, do_sample=False)[0]["generated_text"]
#     return response.strip().replace("Kategória:", "").strip().capitalize()

# # Vygeneruj kategórie
# df["kategoria"] = df.apply(lambda row: create_category(row["criterion"], row["description"]), axis=1)

# # Ulož výstup
# df.to_csv("data/contract_criteria_with_categories.csv", index=False)
# print("Kategórie boli vygenerované a uložené.")

In [7]:
# # Zoskup a spočítaj kategórie
# category_summary = df["kategoria"].value_counts()
# print("Vygenerované kategórie:")
# print(category_summary)

In [14]:
# Príprava TF-IDF vektorov
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['criterion'])
 
# Funkcia na odporúčanie kategórií
def recommend_category(input_text, top_n=3):
    input_vec = vectorizer.transform([input_text])
    similarity_scores = cosine_similarity(input_vec, tfidf_matrix).flatten()
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    recommendations = df.iloc[top_indices][['criterion', 'general_criterion']].copy()
    scores = similarity_scores[top_indices]
    recommendations['similarity'] = scores
    return recommendations
 
# Príklad použitia:
# Zadáte nový text a dostanete odporúčania
input_text = "Celková cena za projekt"  # sem zadajte vaše kritérium
recommendations = recommend_category(input_text)
recommendations

,criterion,general_criterion,similarity
1951,Celková doba výstavby,Ostatné špecifiká,0.603349
1937,Celková doba výstavby,Ostatné špecifiká,0.603349
1890,Celková doba výstavby,Ostatné špecifiká,0.603349


In [17]:
# Výber textových popisov
all_general_criterions = df["general_criterion"].fillna("").astype(str).tolist()

# Výpočet embeddingov
all_embeddings_gc = model.encode(all_general_criterions, convert_to_tensor=True)

# Uloženie
np.save("embeddings/general_criterions_embeddings_split.npy", all_embeddings_gc)
df.to_csv("data/general_criterions_data.csv", index=False)